**问题：综合运用之前所学的内容,构建一个贷款信用风险评估模型,需要做的内容包括但不限于:**
* 数据探索及结论
* 数据清洗
* 数据预处理(编码与特征构造)
* 基于回归模型,将“得分”作为标签进行回归,可以使用不同的模型设计和训练策略,并分析不同的策略对结果的影响
* 基于分类模型,将"逾期"作为标签进行分类,可以使用不同的模型设计和训练策略,并分析不同的策略对结果的影响
* 请完成代码和实验报告

* **出于阅读的简洁性，我将对每一份数据集进行探索-清洗-编码的顺序，依次处理每一份数据集**
* **本文用markdown进行必要的分段，点击大纲即可方便跳转进行阅读**

## 导入必要的包

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn .metrics import accuracy_score, f1_score ##后面还会导入别的

## 数据探索，清洗与预处理

### bank数据集

In [ ]:
bank=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\bank.csv")

In [ ]:
bank.head(2)  ##需要的信息是交易金额和工资收入标记，以便于匹配

In [ ]:
display(bank.info())
display(bank.describe().T)

In [ ]:
bank

In [ ]:
bank.drop('交易时间戳',axis=1,inplace=True)

In [ ]:
bank

#### 处理异常值

In [ ]:
px.box(bank,y='交易金额')

In [ ]:
bank = bank[ bank["交易金额"]>0]
avg_val =bank["交易金额"].mean()
std_val =bank["交易金额"].std()
upper_bound = avg_val + 3 * std_val
lower_bound = avg_val - 3 * std_val
drop_index = (bank ["交易金额"] < lower_bound) | (bank["交易金额"] > upper_bound)
bank = bank[~drop_index]
px.box(bank,y="交易金额")

In [ ]:
bank["交易金额_分箱后"]=pd.cut(bank['交易金额'],3,labels=['低','中','高'])
bank=bank[bank['工资收入标记']==1]  ##只读取是工资收入的记录
bank

In [ ]:
result_incomemean=bank.groupby('new_user_id')['交易金额'].mean()

### overdue数据集

In [ ]:
overdue=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\overdue.csv")

In [ ]:
overdue.head(2)

In [ ]:
overdue=overdue.drop('Unnamed: 0',axis=1)

In [ ]:
print(overdue.info())
print(overdue.describe().T)

### user数据集

In [ ]:
userinfo=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\userinfo.csv")

In [ ]:
userinfo

In [ ]:
print(userinfo.describe().T)
userinfo.info()

#### 编码

In [ ]:
categorical_columns = ['性别', '职业', '教育程度', '婚姻状态', '户口类型'] 
userinfo = pd.get_dummies(userinfo, columns=categorical_columns, prefix=categorical_columns)

In [ ]:
userinfo.head(2)

### bill数据集

In [ ]:
bill=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\bill.csv")

In [ ]:
bill.head(2)

In [ ]:
print(bill.describe().T)
bill.info()

In [ ]:
bill.isnull().sum()

### 初步结论与组合原始

形成对数据集的初步结论
1. 数据集不存在缺失值，数据质量高
2. 数据集整体异常值较少，可靠性强
3. 数据集以信贷风险评估为目标，旨在评估个体的风险，数据集包含了个体的基本信息，银行流水，违约情况，历史还款数据等
4. 经过对各个数据集的初步探索和清洗，下一步构造组合数据集进行进一步的评估

In [ ]:
df=userinfo

In [ ]:
df=df.merge(overdue,on='new_user_id',how='left')

In [ ]:
df = df.merge(result_incomemean.to_frame().rename(columns={'交易金额':'平均交易金额'}), 
              on='new_user_id', 
              how='left')

In [ ]:
df

## 进一步编码与特征生成

这时，要对bill数据集进行更加细致的处理，先前我们对其他三份数据集进行了初步的哑变量编码和分组求值，现在进行特征生成部分

#### 生成状态型特征

In [ ]:
# 基础特征计算
bill["欠款金额"] = bill["上期账单金额"] - bill["上期还款金额"]  # 基础欠款情况
bill["还款比率"] = bill["上期还款金额"] / bill["上期账单金额"]  # 基础还款能力
bill["还款比率"] = bill["还款比率"].replace([np.inf, -np.inf], np.nan).fillna(0)

# 风险相关特征
bill["高额使用率"] = (bill["本期账单余额"] / bill["信用卡额度"] > 0.8).astype(int)  # 高使用率一般更高的违约风险
bill["连续欠款"] = (bill["欠款金额"] > 0) & (bill["上期账单金额"] > 0)  # 连续欠款是违约预警信号
bill["还款压力指数"] = (bill["本期账单最低还款额"] / bill["上期还款金额"]).fillna(0)  # 还款压力越大，违约风险越高
bill["额度消耗速度"] = (bill["信用卡额度"] - bill["可用金额"]) / bill["信用卡额度"]  # 额度消耗过快表示财务压力

agg_features = bill.groupby("new_user_id").agg({
    "还款比率": ["mean", "min", "std"],  # 还款稳定性，std越大表示还款行为越不稳定
    "高额使用率": "mean",  # 高额度使用频率
    "连续欠款": "sum",  # 连续欠款次数
    "还款压力指数": ["mean", "max"],  # 平均和最大还款压力
    "额度消耗速度": ["mean", "max"],  # 额度使用行为
    "欠款金额": ["mean", "max", lambda x: (x > 0).sum()]  # 欠款金额统计和欠款次数
}).round(4)

agg_features.columns = [
    "平均还款比率", "最低还款比率", "还款波动性",
    "高额度使用频率",
    "连续欠款次数",
    "平均还款压力", "最大还款压力",
    "平均额度消耗率", "最大额度消耗率",
    "平均欠款金额", "最大欠款金额", "欠款次数"
]
agg_features = agg_features.fillna(agg_features.mean())
agg_features

In [ ]:
bill.head(2)

#### 生成趋势型特征

* 趋势特征与时间序列有关，下面进入时间序列的分析

In [ ]:
bank_train=bank
bank_train['交易时间'].to_timestamp
tmp = bank_train.groupby("new_user_id")[["交易金额","交易时间"]]

In [ ]:
def get_recent_data(df, time_col, n, unit='M'):
    """
    提取时间序列数据中最近n个时间单位的数据
    
    参数:
    df: DataFrame - 输入的数据框
    time_col: str - 时间列的名称
    n: int - 要提取的时间单位数量
    unit: str - 时间单位('D'天,'W'周,'M'月,'Y'年),默认为'M'月
    """

    df[time_col] = pd.to_datetime(df[time_col])  ##转化为时间戳数据类型
    max_date = df[time_col].max()  ##最近的日期
    
    if unit == 'M':
        start_date = max_date - pd.DateOffset(months=n)  ##优化原函数，使用datatime的日期偏置简化函数
    elif unit == 'D':
        start_date = max_date - pd.DateOffset(days=n)
    elif unit == 'W':
        start_date = max_date - pd.DateOffset(weeks=n)
    elif unit == 'Y':
        start_date = max_date - pd.DateOffset(years=n)
    else:
        raise ValueError("不支持的时间单位,请使用 'D','W','M' 或 'Y'")
    
    filtered_df = df[df[time_col] >= start_date]
    return filtered_df


In [ ]:
import datetime
from dateutil.relativedelta import relativedelta
# 计算6个月的数据统计
recent_data_6 = tmp.apply(lambda x: get_recent_data(x, '交易时间', 6, 'M'))
recent_data_6 = recent_data_6.groupby("new_user_id")[["交易金额"]].sum()
recent_data_6.columns = ["6月内工资收入"]
recent_data_6

In [ ]:
# 计算3个月的数据统计
recent_data_3 = tmp.apply(lambda x: get_recent_data(x, '交易时间', 3, 'M'))
recent_data_3 = recent_data_3.groupby("new_user_id")[["交易金额"]].sum()
recent_data_3.columns = ["3月内工资收入"]
recent_data_3 = recent_data_3.reset_index()
recent_data_3

In [ ]:
# 合并数据并计算趋势
trend_df = pd.merge(recent_data_6, recent_data_3, how='left', on=['new_user_id'])
trend_df["半年到3月前工资收入"] = trend_df["6月内工资收入"] - trend_df['3月内工资收入']
trend_df["过去半年内工资增长趋势"] = (trend_df['3月内工资收入'] >= trend_df['半年到3月前工资收入']).astype(int)
trend_df

#### 生成统计型特征

In [ ]:
# 基础统计特征
stats_features = bank.groupby('new_user_id').agg({
    '交易金额': ['max', 'min', 'std', 'mean']  # 收入的基础统计量
}).round(4)

stats_features.columns = [
    '最高月收入', '最低月收入', '收入波动性', '平均月收入'
]


##### 计算月度波动量

In [ ]:
# 创建一个空的DataFrame来存储所有月份的数据
result_df = pd.DataFrame()

# 循环计算每个月的数据
for month in range(1, 7):
    month_data = tmp.apply(lambda x: get_recent_data(x, '交易时间', month, 'M'))
    month_data = month_data.groupby("new_user_id")[["交易金额"]].sum()
    # 重命名列名
    month_data.columns = [f"{month}月内平均收入"]
    # 如果是第一个月，直接赋值给result_df
    if month == 1:
        result_df = month_data
    # 否则，合并数据
    else:
        result_df = result_df.join(month_data)

# 计算月度变化并求标准差
result_df['波动量'] = result_df.apply(
    lambda x: np.std([
        x[f'{i+1}月内平均收入'] - x[f'{i}月内平均收入'] 
        for i in range(1, 6)
    ]), 
    axis=1
)

In [ ]:
result_df=result_df['波动量']
result_df

In [ ]:
stats_features=stats_features.merge(result_df,on='new_user_id',how='left')
stats_features

## 训练前的准备：合并最终数据集与归一化

### 组合数据集

In [ ]:
df_merged = stats_features.merge(df, on='new_user_id', how='left')\
              .merge(trend_df, on='new_user_id', how='left')\
              .merge(agg_features, on='new_user_id', how='left')

In [ ]:
df_merged

## 训练模型

In [ ]:
'''features = [
    # 用户基本信息
    '性别_女', '性别_未知', '性别_男',
    '职业_事业单位', '职业_企业', '职业_公职', '职业_其他', '职业_未知',
    '教育程度_大专', '教育程度_本科及以上', '教育程度_未知', '教育程度_高中及以下',
    '婚姻状态_已婚', '婚姻状态_未婚', '婚姻状态_未知', '婚姻状态_离异',
    '户口类型_其他户口', '户口类型_农业户口', '户口类型_家庭户口', '户口类型_未知', '户口类型_集体户口',
    
    # 账单特征统计量
    '平均还款比率', '最低还款比率', '还款波动性',
    '高额度使用频率', '连续欠款次数',
    '平均还款压力', '最大还款压力',
    '平均额度消耗率', '最大额度消耗率',
    '平均欠款金额', '最大欠款金额', '欠款次数',
    
   ' # 工资收入趋势
    '6月内工资收入', '3月内工资收入',
    '半年到3月前工资收入', '过去半年内工资增长趋势',
    
    # 收入统计特征
    '最高月收入', '最低月收入', '收入波动性', '平均月收入', '波动量'
]
'''

#### 使用回归模型

##### 分离特征和标签

In [ ]:
from sklearn.preprocessing import StandardScaler
y = df_merged['分数']
X = df_merged[[  
    '性别_女', '性别_男',
    '职业_事业单位', '职业_企业', '职业_公职', 
    '教育程度_大专', '教育程度_本科及以上', '教育程度_未知', 
    '婚姻状态_已婚', '婚姻状态_未婚', '婚姻状态_未知', '婚姻状态_离异',  ## 基本特征
    '户口类型_其他户口', '户口类型_农业户口', '户口类型_家庭户口', '户口类型_集体户口',
    '平均还款比率','平均欠款金额','欠款次数','还款波动性', ##状态特征选取
    '过去半年内工资增长趋势', ##趋势特征
    '平均月收入', '波动量' ##统计特征
    #在这里，我们选取的主要原理是根据基本情况+根据直觉选中的，在优化部门，我们将选取最优的模型
]]

标准化

In [ ]:
numeric_columns = X.select_dtypes(include=['float64', 'int64']).columns
y = StandardScaler().fit_transform(y.values.reshape(-1, 1)).ravel()
categorical_columns = X.select_dtypes(include=['bool', 'object']).columns
X_numeric = X[numeric_columns]
X_categorical = X[categorical_columns]
X_numeric = X_numeric.fillna(X_numeric.mean())
X_categorical = X_categorical.fillna(X_categorical.mode().iloc[0])
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)
X_numeric_scaled = pd.DataFrame(X_numeric_scaled, columns=X_numeric.columns)
X_processed = pd.concat([X_numeric_scaled, X_categorical], axis=1)
X=X_processed

拆分训练集和测试集


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 训练和评估模型
models = {
    "线性回归": LinearRegression(),
    "岭回归": Ridge(alpha=10), ##这里的超参数在于正则化强度
    "Lasso回归": Lasso(alpha=0.1),
    "随机森林回归": RandomForestRegressor(n_estimators=100, random_state=42)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)  # 使用正确的训练集
    y_pred = model.predict(X_test)  # 使用正确的测试集
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results[name] = {
        "RMSE": rmse,
        "R2": r2
    }

# 打印结果
print("回归模型性能比较：")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"  RMSE: {metrics['RMSE']:.4f}")
    print(f"  R2: {metrics['R2']:.4f}")
    print()


In [ ]:
# 选择最佳模型（以R2为标准）
best_model = max(results, key=lambda x: results[x]['R2'])
print(f"最佳模型: {best_model}")

In [ ]:
rf_model = models["随机森林回归"]
feature_importance = pd.DataFrame({
    'feature': X.columns,  # 使用正确的特征列名
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
    
print("\n随机森林回归 - 前10个最重要特征:")
feature_importance.head(10)

#### 使用分类模型

In [ ]:
# 导入必要的分类模型和评估指标
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [ ]:
###数据集
y_class = df_merged['标签']
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X, 
    y_class,
    test_size=0.2,
    random_state=42
)

In [ ]:
# 定义分类模型
classification_models = {
    "逻辑回归": LogisticRegression(random_state=42, C=0.01, penalty='l1',solver= 'liblinear'),
    "决策树": DecisionTreeClassifier(random_state=42),
    "随机森林分类": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42)  ##一般svm对数据要求比较高，因为用的是500多个样本所以能训练完，
}

classification_results = {}
for name, model in classification_models.items():
    model.fit(X_train_class, y_train_class)
    y_pred = model.predict(X_test_class)
    classification_results[name] = {
        "准确率": accuracy_score(y_test_class, y_pred),
        "精确率": precision_score(y_test_class, y_pred),
        "召回率": recall_score(y_test_class, y_pred),
        "F1分数": f1_score(y_test_class, y_pred)
    }

In [ ]:
print("分类模型性能比较：")
for name, metrics in classification_results.items():
    print(f"\n{name}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

# 选择最佳模型（以F1分数为标准）
best_class_model = max(classification_results, key=lambda x: classification_results[x]['F1分数'])
print(f"\n最佳分类模型: {best_class_model}")

In [ ]:
rf_class_model = classification_models["随机森林分类"]
feature_importance_class = pd.DataFrame({
    'feature': X_processed.columns,
    'importance': rf_class_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\n随机森林分类器 - 前10个最重要特征:")
feature_importance_class.head(10)

## 优化模型

#### 超参数选取

1. 这里的超参数主要利用交叉验证法验证最优的正则化系数值
2. 以及利用网络搜索法寻找最优情况
   1. 然后利用网格法有时候会无法收敛，实测输出的最优效果提升有限甚至负提升
   2. 因此在开始的回归model中有选择的修改超参数
   3. 更多的超参数含义因为模型不太熟练而没有多用

In [ ]:
##采用交叉验证法确定最优的的岭回归正则化系数值
from sklearn.model_selection import GridSearchCV
param_grid = {'alpha': [0.1, 0.5, 1.0, 10.0]}
ridge = Ridge()
grid_search = GridSearchCV(ridge, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)
print(f"最佳alpha: {grid_search.best_params_['alpha']}")

In [ ]:
##采用交叉验证法确定最优的的 lasso回归正则化系数值
param_grid = {'alpha': [0.1, 0.5, 1.0, 10.0], 'tol': [1e-4, 1e-3, 1e-2]}
lasso = Lasso()
grid_search = GridSearchCV(lasso, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)
print(f"最佳alpha: {grid_search.best_params_['alpha']}, tol: {grid_search.best_params_['tol']}")

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}
log_reg = LogisticRegression(random_state=42)
grid_search = GridSearchCV(log_reg, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train_class, y_train_class)
print(f"最佳参数: {grid_search.best_params_}")

In [ ]:
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}
dt = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train_class, y_train_class)
print(f"最佳参数: {grid_search.best_params_}")

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 1, 10],
    'kernel': ['rbf', 'poly', 'sigmoid']
}
svc = SVC(random_state=42)
grid_search = GridSearchCV(svc, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train_class, y_train_class)
print(f"最佳参数: {grid_search.best_params_}")

#### 换样本

* 在上述模型中，利用的是500多个样本的全数据集，这就导致损失了大量样本没有得到利用，因此这里采用全样本进行训练，具体的代码与前文完全一致，不过多解释

In [ ]:
df_all=df.merge(stats_features, on='new_user_id', how='left')\
              .merge(trend_df, on='new_user_id', how='left')\
              .merge(agg_features, on='new_user_id', how='left')

In [ ]:
df_all

In [ ]:
from sklearn.preprocessing import StandardScaler
y = df_all['分数']
X = df_all[[  
    '性别_女', '性别_男',
    '职业_事业单位', '职业_企业', '职业_公职', 
    '教育程度_大专', '教育程度_本科及以上', '教育程度_未知', 
    '婚姻状态_已婚', '婚姻状态_未婚', '婚姻状态_未知', '婚姻状态_离异',  ## 基本特征
    '户口类型_其他户口', '户口类型_农业户口', '户口类型_家庭户口', '户口类型_集体户口',
    '平均还款比率','平均欠款金额','欠款次数','还款波动性', ##状态特征选取
    '过去半年内工资增长趋势', ##趋势特征
    '平均月收入', '波动量' ##统计特征
]]
numeric_columns = X.select_dtypes(include=['float64', 'int64']).columns
y = StandardScaler().fit_transform(y.values.reshape(-1, 1)).ravel()
categorical_columns = X.select_dtypes(include=['bool', 'object']).columns
X_numeric = X[numeric_columns]
X_categorical = X[categorical_columns]
X_numeric = X_numeric.fillna(X_numeric.mean())
X_categorical = X_categorical.fillna(X_categorical.mode().iloc[0])
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)
X_numeric_scaled = pd.DataFrame(X_numeric_scaled, columns=X_numeric.columns)
X_processed = pd.concat([X_numeric_scaled, X_categorical], axis=1)
X=X_processed
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.2,
    random_state=40
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 训练和评估模型
models = {
    "线性回归": LinearRegression(),
    "岭回归": Ridge(alpha=10), ##这里的超参数在于正则化强度
    "Lasso回归": Lasso(alpha=0.1),
    "随机森林回归": RandomForestRegressor(n_estimators=100, random_state=42)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)  # 使用正确的训练集
    y_pred = model.predict(X_test)  # 使用正确的测试集
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results[name] = {
        "RMSE": rmse,
        "R2": r2
    }

# 打印结果
print("回归模型性能比较：")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"  RMSE: {metrics['RMSE']:.4f}")
    print(f"  R2: {metrics['R2']:.4f}")
    print()
# 选择最佳模型（以R2为标准）
best_model = max(results, key=lambda x: results[x]['R2'])
print(f"最佳模型: {best_model}")
rf_model = models["随机森林回归"]
feature_importance = pd.DataFrame({
    'feature': X.columns,  # 使用正确的特征列名
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
    
print("\n随机森林回归 - 前10个最重要特征:")
feature_importance.head(10)

In [ ]:
###数据集
y_class = df_all['标签']
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X, 
    y_class,
    test_size=0.2,
    random_state=40
)
# 定义分类模型
classification_models = {
    "逻辑回归": LogisticRegression(random_state=42, C=0.01, penalty='l1',solver= 'liblinear'),
    "决策树": DecisionTreeClassifier(random_state=42),
    "随机森林分类": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42)  ##一般svm对数据要求比较高，因为用的是500多个样本所以能训练完，
}

classification_results = {}
for name, model in classification_models.items():
    model.fit(X_train_class, y_train_class)
    y_pred = model.predict(X_test_class)
    classification_results[name] = {
        "准确率": accuracy_score(y_test_class, y_pred),
        "精确率": precision_score(y_test_class, y_pred),
        "召回率": recall_score(y_test_class, y_pred),
        "F1分数": f1_score(y_test_class, y_pred)
    }
print("分类模型性能比较：")
for name, metrics in classification_results.items():
    print(f"\n{name}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

# 选择最佳模型（以F1分数为标准）
best_class_model = max(classification_results, key=lambda x: classification_results[x]['F1分数'])
print(f"\n最佳分类模型: {best_class_model}")
print("分类模型性能比较：")
for name, metrics in classification_results.items():
    print(f"\n{name}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

# 选择最佳模型（以F1分数为标准）
best_class_model = max(classification_results, key=lambda x: classification_results[x]['F1分数'])
print(f"\n最佳分类模型: {best_class_model}")

发现在大样本下，训练效果有优有劣，因此还是以小样本为准

#### 换特征

利用回归森林，其实可以选择最优的特征，现在把全特征全部进入进去，选取最优的15个特征进行回归

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

###统统加入，除了id和二值变量全放进去
y = df_all['分数']
X = df_all.drop(['分数','new_user_id','标签'], axis=1)

# 把一堆变量转化为哑变量
for column in X.columns:
    if X[column].dtype == 'object':
        le = LabelEncoder()
        X[column] = le.fit_transform(X[column])  # 将字符串转换为整数标签

# 常规流程，直接掉包
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"模型性能：")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

In [ ]:
# 获取特征重要性并选择最重要的15个特征
feature_importances = pd.DataFrame(rf.feature_importances_, index=X_train.columns, columns=['importance']).sort_values('importance', ascending=False)
top_15_features = feature_importances.head(15).index.tolist()
X_train_top15 = X_train[top_15_features]
X_test_top15 = X_test[top_15_features]
print
rf_top15 = RandomForestRegressor(n_estimators=100, random_state=42)
rf_top15.fit(X_train_top15, y_train)
y_pred = rf_top15.predict(X_test_top15)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"使用最重要的15个特征的模型性能：")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")


In [ ]:
feature_importances_top15 = pd.DataFrame(rf_top15.feature_importances_, index=X_train_top15.columns, columns=['importance']).sort_values('importance', ascending=False)
print("最重要的15个特征及其重要性评分：")
print(feature_importances_top15)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 准备数据
y_class = df_merged['标签']
X_class = df_merged.drop(['标签','new_user_id','分数'], axis=1)

# 处理数值型特征
numeric_columns = X_class.select_dtypes(include=['float64', 'int64']).columns
categorical_columns = X_class.select_dtypes(include=['bool', 'object']).columns

# 标准化数值特征
X_numeric = X_class[numeric_columns]
X_categorical = X_class[categorical_columns]
X_numeric = X_numeric.fillna(X_numeric.mean())
X_categorical = X_categorical.fillna(X_categorical.mode().iloc[0])

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)
X_numeric_scaled = pd.DataFrame(X_numeric_scaled, columns=X_numeric.columns)
X_processed = pd.concat([X_numeric_scaled, X_categorical], axis=1)

# 拆分数据集
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_processed, y_class, test_size=0.2, random_state=42
)

# 训练随机森林分类器获取特征重要性
rf_class = RandomForestClassifier(n_estimators=100, random_state=42)
rf_class.fit(X_train_class, y_train_class)

# 获取并显示特征重要性
feature_importance_class = pd.DataFrame({
    'feature': X_processed.columns,
    'importance': rf_class.feature_importances_
}).sort_values('importance', ascending=False)

# 选择top 15特征
top_15_features = feature_importance_class.head(15)['feature'].tolist()
X_train_top15 = X_train_class[top_15_features]
X_test_top15 = X_test_class[top_15_features]

# 使用top 15特征重新训练模型
rf_class_top15 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_class_top15.fit(X_train_top15, y_train_class)
y_pred = rf_class_top15.predict(X_test_top15)

# 评估模型
results = {
    "准确率": accuracy_score(y_test_class, y_pred),
    "精确率": precision_score(y_test_class, y_pred),
    "召回率": recall_score(y_test_class, y_pred),
    "F1分数": f1_score(y_test_class, y_pred)
}

# 打印结果
print("使用Top 15特征的分类模型性能：")
for metric_name, value in results.items():
    print(f"  {metric_name}: {value:.4f}")

print("\n最重要的15个特征及其重要性：")
print(feature_importance_class.head(15))

综合来看，选取的15个特征可以较好的描述模型，相比于全部的50个特征有更好的可解释性和简洁性